# 02 — Loading Remaining Datasets
## SBDR Phase A, Step A2
UCI is done (NB01). Now we load the other 4 datasets, do quick health checks, and save processed versions.

In [2]:
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.4/810.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 32.4 MB/s eta 0:00:00m eta 0:00:010:01:01


In [3]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports loaded.")

All imports loaded.


## Dataset 1: Lending Club (2.26M loans)
Using Polars for this one — too large for Pandas to handle efficiently. We only load the columns relevant to our project.

In [4]:
# Columns we need for SBDR
lc_cols = [
    'loan_status', 'loan_amnt', 'funded_amnt', 'annual_inc', 
    'dti', 'revol_util', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'purpose',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc',
    'pub_rec', 'total_acc', 'installment', 'int_rate'
]

# Lazy scan — doesn't load into memory until we collect()
lc = pl.scan_csv(
    "../data/raw/LendingClub_accepted2007_2018.csv",
    ignore_errors=True
).select(lc_cols).collect()

print(f"Shape: {lc.shape}")
print(f"Columns: {lc.columns}")
lc.head()

Shape: (2260701, 18)
Columns: ['loan_status', 'loan_amnt', 'funded_amnt', 'annual_inc', 'dti', 'revol_util', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'purpose', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc', 'installment', 'int_rate']


loan_status,loan_amnt,funded_amnt,annual_inc,dti,revol_util,grade,sub_grade,emp_length,home_ownership,purpose,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,total_acc,installment,int_rate
str,f64,f64,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64
"""Fully Paid""",3600.0,3600.0,55000.0,5.91,29.7,"""C""","""C4""","""10+ years""","""MORTGAGE""","""debt_consolidation""",0.0,1.0,7.0,0.0,13.0,123.03,13.99
"""Fully Paid""",24700.0,24700.0,65000.0,16.06,19.2,"""C""","""C1""","""10+ years""","""MORTGAGE""","""small_business""",1.0,4.0,22.0,0.0,38.0,820.28,11.99
"""Fully Paid""",20000.0,20000.0,63000.0,10.78,56.2,"""B""","""B4""","""10+ years""","""MORTGAGE""","""home_improvement""",0.0,0.0,6.0,0.0,18.0,432.66,10.78
"""Current""",35000.0,35000.0,110000.0,17.06,11.6,"""C""","""C5""","""10+ years""","""MORTGAGE""","""debt_consolidation""",0.0,0.0,13.0,0.0,17.0,829.9,14.85
"""Fully Paid""",10400.0,10400.0,104433.0,25.37,64.5,"""F""","""F1""","""3 years""","""MORTGAGE""","""major_purchase""",1.0,3.0,12.0,0.0,35.0,289.91,22.45


In [5]:
# Data types
print("=== Data Types ===")
print(lc.dtypes)
print(f"\n=== Null Counts ===")
print(lc.null_count())

=== Data Types ===
[String, Float64, Float64, Float64, Float64, Float64, String, String, String, String, String, Float64, Float64, Float64, Float64, Float64, Float64, Float64]

=== Null Counts ===
shape: (1, 18)
┌────────────┬───────────┬────────────┬───────────┬───┬─────────┬───────────┬───────────┬──────────┐
│ loan_statu ┆ loan_amnt ┆ funded_amn ┆ annual_in ┆ … ┆ pub_rec ┆ total_acc ┆ installme ┆ int_rate │
│ s          ┆ ---       ┆ t          ┆ c         ┆   ┆ ---     ┆ ---       ┆ nt        ┆ ---      │
│ ---        ┆ u32       ┆ ---        ┆ ---       ┆   ┆ u32     ┆ u32       ┆ ---       ┆ u32      │
│ u32        ┆           ┆ u32        ┆ u32       ┆   ┆         ┆           ┆ u32       ┆          │
╞════════════╪═══════════╪════════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪══════════╡
│ 33         ┆ 33        ┆ 33         ┆ 37        ┆ … ┆ 62      ┆ 62        ┆ 33        ┆ 33       │
└────────────┴───────────┴────────────┴───────────┴───┴─────────┴───────────┴────

In [7]:
# What loan outcomes exist?
status_counts = lc.group_by('loan_status').agg(
    pl.len().alias('count')
).sort('count', descending=True)

print(status_counts)

shape: (10, 2)
┌─────────────────────────────────┬─────────┐
│ loan_status                     ┆ count   │
│ ---                             ┆ ---     │
│ str                             ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ Fully Paid                      ┆ 1076751 │
│ Current                         ┆ 878317  │
│ Charged Off                     ┆ 268559  │
│ Late (31-120 days)              ┆ 21467   │
│ In Grace Period                 ┆ 8436    │
│ Late (16-30 days)               ┆ 4349    │
│ Does not meet the credit polic… ┆ 1988    │
│ Does not meet the credit polic… ┆ 761     │
│ Default                         ┆ 40      │
│ null                            ┆ 33      │
└─────────────────────────────────┴─────────┘


### Loan Status → SBDR Tier Mapping
We map Lending Club's loan outcomes to our 4-tier recovery system. This gives us real-world labels for how customers progress through financial distress.